# Phase 5: Probabilistic Forecasting and Uncertainty Quantification

This implements a new LOCF-based probabilistic policy, separate from the frozen Phase 4 point policy. Source-specific lag assumptions and revised-snapshot limitations remain. No advanced mean models, BOM costing, fabricated material mappings or pricing optimization are introduced. All data and generated artifacts remain local and Git-ignored.

In [ ]:
import sys
from pathlib import Path

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").exists()
)
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "scripts"))
from run_phase5 import main

results = main()
import pandas as pd
from IPython.display import Image, display

out = results["output"]

## Random-walk diagnostics

Validated ADF/KPSS and ACF/PACF existed in Phase 2. Phase 5 adds Ljung–Box, heteroskedasticity-robust variance ratios, HAC drift tests, ARCH-LM and structural-break sensitivity. Holm controls multiplicity across the full battery; BH family results are exploratory. Each test has a different null. Non-rejection is not proof of a random walk, independent innovations or stable volatility.

In [ ]:
display(results["diagnostics"])
print((out / "random_walk_diagnostic_report.md").read_text())
display(Image(filename=str(out / "07_random_walk_diagnostic_summary.png")))
display(Image(filename=str(out / "08_return_acf_pacf.png")))

## Probabilistic protocol and evaluation

2020–2021 forecast origins select core methods, freezing December 31, 2022. Dedicated 2022–2023 LOCF forecast errors calibrate conformal intervals. Monthly confirmation starts January 2025 and only scores labels available by the final September decision. Error pools exclude confirmation-origin residuals. Parameters based on historical innovations update only from then-observable history. GARCH is attempted only after a training-only ARCH/FDR gate and stability checks. Conformal finite-sample ranks do not establish exact coverage under dependent, shifting time series.

In [ ]:
display(results["scores"])
display(results["macro"])
display(results["choices"])
display(pd.read_csv(out / "challenger_matched_performance.csv"))
display(pd.read_csv(out / "coverage_dependence_uncertainty.csv"))
for name in (
    "03_prediction_interval_coverage",
    "04_quantile_calibration",
    "05_pinball_and_sharpness",
    "06_historical_conditional_volatility",
):
    display(Image(filename=str(out / f"{name}.png")))

## Final probabilistic forecasts

December 2026, March 2027 and June 2027 require six, nine and twelve months from the last eligible June observations. P50 need not equal LOCF under asymmetric forecast errors. P10/P90 form a nominal central 80% interval; actual calibration is evaluated, not assumed. Expected prices are reported only when a distribution is defined. Connecting fan-chart lines do not imply additional intermediate forecasts.

In [ ]:
display(results["final"].loc[results["final"].selected])
for path in sorted(out.glob("01_fan_*.png")):
    display(Image(filename=str(path)))
display(Image(filename=str(out / "02_quantiles_by_material_horizon.png")))

## Joint material and FX dependence

Shared blocks of aligned innovations preserve relevant historical dependence. Joint scenarios deliberately synchronize all anchors to June and discard the extra July FX observation. EUR prices are calculated pathwise from USD prices and the shared FX path. These empirical joint scenarios are not reconciled to all selected univariate marginals. No actual BOM cost is estimated, and summing individual P90s would not give the P90 of a weighted total.

In [ ]:
display(pd.read_csv(out / "joint_material_fx_scenarios.csv").head())
display(Image(filename=str(out / "09_joint_innovation_dependence.png")))
print((out / "probabilistic_forecasting_report.md").read_text())